In [1]:
# ==================== SETUP & IMPORT ====================
import os
import numpy as np
import pandas as pd
import json
import gc
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam

# Path
ABLATION_DIR = "/kaggle/working/ablation_study"
os.makedirs(ABLATION_DIR, exist_ok=True)

IMG_SIZE = 384
BATCH_SIZE = 8
SEED = 42
EPOCHS = 50

np.random.seed(SEED)
print("Setup selesai.")

2026-06-09 19:00:53.580086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781031653.762713      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781031653.815487      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781031654.264853      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781031654.264896      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781031654.264902      22 computation_placer.cc:177] computation placer alr

Setup selesai.


In [2]:
# ==================== DATA PIPELINE ====================
import glob
import tensorflow as tf

DATA_ROOT = "/kaggle/input/datasets/lihuayang111265/flood-semantic-segmentation-dataset/dataset/"

train_images = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "images", "*.png")))
train_labels = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "labels", "*.png")))
val_images   = sorted(glob.glob(os.path.join(DATA_ROOT, "val", "images", "*.png")))
val_labels   = sorted(glob.glob(os.path.join(DATA_ROOT, "val", "labels", "*.png")))

def load_pair(image_path, mask_path):
    img = tf.io.read_file(image_path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='bilinear')
    img = tf.cast(img, tf.float32)
    
    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method='nearest')
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def augment_train(img, mask):
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = preprocess_input(img)
    return img, mask

def preprocess_val(img, mask):
    return preprocess_input(img), mask

train_ds = (tf.data.Dataset.from_tensor_slices((train_images, train_labels))
            .shuffle(len(train_images), seed=SEED)
            .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
            .map(augment_train, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((val_images, val_labels))
          .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
          .map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

I0000 00:00:1781031668.905111      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781031668.911036      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# ==================== LOSS & METRICS (DIPERBAIKI) ====================
SMOOTH = 1e-6

def dice_coef(y_true, y_pred):
    yt = K.flatten(y_true)
    yp = K.flatten(y_pred)
    inter = K.sum(yt * yp)
    return (2. * inter + SMOOTH) / (K.sum(yt) + K.sum(yp) + SMOOTH)

def dice_loss(y_true, y_pred):          # INI YANG BENAR
    return 1.0 - dice_coef(y_true, y_pred)

def focal_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, gamma=1.33):
    yt = K.flatten(y_true)
    yp = K.flatten(y_pred)
    tp = K.sum(yt * yp)
    fn = K.sum(yt * (1 - yp))
    fp = K.sum((1 - yt) * yp)
    tversky = (tp + SMOOTH) / (tp + alpha * fp + beta * fn + SMOOTH)
    return K.pow(1 - tversky, gamma)

def iou_metric(y_true, y_pred):
    yt = K.flatten(y_true)
    yp = K.flatten(K.cast(y_pred > 0.5, tf.float32))
    inter = K.sum(yt * yp)
    union = K.sum(yt) + K.sum(yp) - inter
    return (inter + SMOOTH) / (union + SMOOTH)

In [4]:
# ==================== MODEL ====================
def attention_gate(x, g, inter_channels):
    theta_x = layers.Conv2D(inter_channels, 1, padding='same')(x)
    phi_g = layers.Conv2D(inter_channels, 1, padding='same')(g)
    f = layers.Activation('relu')(layers.add([theta_x, phi_g]))
    psi = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(f)
    return layers.multiply([x, psi])

def decoder_block(x, skip, filters, use_attention=True):
    x = layers.UpSampling2D((2, 2), interpolation='bilinear')(x)
    if use_attention:
        skip = attention_gate(skip, x, filters // 2)
    x = layers.Concatenate()([x, skip])
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    return x

def build_model(use_attention=True):
    base = ResNet50(input_shape=[IMG_SIZE, IMG_SIZE, 3], include_top=False, weights='imagenet')
    base.trainable = False
    s1 = base.get_layer('conv1_relu').output
    s2 = base.get_layer('conv2_block3_out').output
    s3 = base.get_layer('conv3_block4_out').output
    s4 = base.get_layer('conv4_block6_out').output
    b  = base.get_layer('conv5_block3_out').output

    d4 = decoder_block(b, s4, 256, use_attention)
    d3 = decoder_block(d4, s3, 128, use_attention)
    d2 = decoder_block(d3, s2, 64, use_attention)
    d1 = decoder_block(d2, s1, 32, use_attention)
    d0 = layers.UpSampling2D((2, 2), interpolation='bilinear')(d1)
    d0 = layers.Conv2D(32, 3, padding='same', use_bias=False)(d0)
    d0 = layers.BatchNormalization()(d0)
    d0 = layers.Activation('relu')(d0)
    out = layers.Conv2D(1, 1, activation='sigmoid')(d0)
    return Model(inputs=base.input, outputs=out)

In [5]:
# ==================== TRAINING + EVALUASI LENGKAP ====================
def train_ablation_variant(variant_name, use_attention, loss_fn, loss_name):
    variant_dir = os.path.join(ABLATION_DIR, variant_name)
    os.makedirs(variant_dir, exist_ok=True)
    metrics_path = os.path.join(variant_dir, 'metrics.json')

    if os.path.exists(metrics_path):
        print(f"[SKIP] {variant_name} sudah ada.")
        return json.load(open(metrics_path))

    tf.keras.backend.clear_session()
    np.random.seed(SEED)

    model = build_model(use_attention=use_attention)
    model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=[dice_coef, iou_metric])

    # Phase 1
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1,
                   callbacks=[EarlyStopping('val_iou_metric', mode='max', patience=8, restore_best_weights=True, verbose=1),
                              ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)])

    # Phase 2
    for layer in model.layers:
        layer.trainable = True
    model.optimizer.learning_rate.assign(1e-5)
    model.compile(optimizer=Adam(1e-5), loss=loss_fn, metrics=[dice_coef, iou_metric])

    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1,
                   callbacks=[EarlyStopping('val_iou_metric', mode='max', patience=12, restore_best_weights=True, verbose=1),
                              ReduceLROnPlateau('val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)])

    # Evaluasi lengkap
    preds_all, masks_all = [], []
    for imgs, msks in val_ds:
        preds_all.append(model.predict(imgs, verbose=0))
        masks_all.append(msks.numpy())

    preds_all = np.concatenate(preds_all, 0).squeeze()
    masks_all = np.concatenate(masks_all, 0).squeeze()

    # Threshold search
    best_iou, best_th = 0, 0.5
    for th in np.arange(0.25, 0.80, 0.05):
        pb = (preds_all > th).astype(np.uint8)
        iou = np.logical_and(pb, masks_all).sum() / (pb.sum() + masks_all.sum() - np.logical_and(pb, masks_all).sum() + 1e-7)
        if iou > best_iou:
            best_iou, best_th = iou, th

    # Hitung metrik lengkap
    f1_list, prec_list, rec_list = [], [], []
    for i in range(len(preds_all)):
        p = (preds_all[i] > best_th).astype(np.uint8)
        g = masks_all[i].astype(np.uint8)
        tp = np.logical_and(p == 1, g == 1).sum()
        fp = np.logical_and(p == 1, g == 0).sum()
        fn = np.logical_and(p == 0, g == 1).sum()

        f1   = 2 * tp / (2 * tp + fp + fn + 1e-7) if (2 * tp + fp + fn) > 0 else 0.0
        prec = tp / (tp + fp + 1e-7)              if (tp + fp) > 0          else 0.0
        rec  = tp / (tp + fn + 1e-7)              if (tp + fn) > 0          else 0.0

        f1_list.append(f1)
        prec_list.append(prec)
        rec_list.append(rec)

    # Simpan
    np.save(os.path.join(variant_dir, 'per_image_f1.npy'),   np.array(f1_list))
    np.save(os.path.join(variant_dir, 'per_image_prec.npy'), np.array(prec_list))
    np.save(os.path.join(variant_dir, 'per_image_rec.npy'),  np.array(rec_list))
    model.save_weights(os.path.join(variant_dir, 'model.weights.h5'))

    summary = {
        'variant': variant_name,
        'use_attention': use_attention,
        'loss': loss_name,
        'mean_f1': float(np.mean(f1_list)),
        'mean_prec': float(np.mean(prec_list)),
        'mean_rec': float(np.mean(rec_list)),
        'agg_iou': float(best_iou),
        'threshold': float(best_th),
        'phase1_epochs': len(h1.history['loss']),
        'phase2_epochs': len(h2.history['loss'])
    }

    with open(metrics_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"[{variant_name}] F1={summary['mean_f1']*100:.2f}% | "
          f"Prec={summary['mean_prec']*100:.2f}% | Rec={summary['mean_rec']*100:.2f}% | "
          f"IoU={summary['agg_iou']*100:.2f}%")

    return summary

In [6]:
# ==================== JALANKAN SEMUA VARIANT ====================
print("\n" + "="*70)
print("MENJALANKAN ABLATION STUDY LENGKAP (A, B, C, D)")
print("="*70)

variants = [
    ("Variant_A_NoAtt_Dice",        False, dice_loss,           "Dice Loss"),
    ("Variant_B_NoAtt_FocalTversky", False, focal_tversky_loss, "Focal Tversky"),
    ("Variant_C_Att_Dice",           True,  dice_loss,           "Dice Loss"),
    ("Variant_D_Att_FocalTversky",   True,  focal_tversky_loss, "Focal Tversky"),
]

results = []
for name, use_att, loss_fn, loss_name in variants:
    print(f"\n>>> Training {name}")
    try:
        res = train_ablation_variant(name, use_att, loss_fn, loss_name)
        results.append(res)
    except Exception as e:
        print(f"[ERROR] {name}: {e}")
        gc.collect()

# Simpan ringkasan
with open(os.path.join(ABLATION_DIR, "ablation_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*70)
print("SELESAI — Semua variant sudah selesai dengan metrik lengkap")
print("="*70)


MENJALANKAN ABLATION STUDY LENGKAP (A, B, C, D)

>>> Training Variant_A_NoAtt_Dice
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/50


I0000 00:00:1781031685.509481      68 service.cc:152] XLA service 0x7c1e24002a00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781031685.509526      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1781031685.509531      68 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1781031688.178422      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-09 19:01:38.669838: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-09 19:01:38.962230: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-09 19:01:39.924037: E external/local_xl

75/75 ━━━━━━━━━━━━━━━━━━━━ 76s 478ms/step - dice_coef: 0.8573 - iou_metric: 0.8207 - loss: 0.1427 - val_dice_coef: 0.7246 - val_iou_metric: 0.6269 - val_loss: 0.2741 - learning_rate: 0.0010
Epoch 2/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 20s 269ms/step - dice_coef: 0.9200 - iou_metric: 0.8760 - loss: 0.0800 - val_dice_coef: 0.9227 - val_iou_metric: 0.8899 - val_loss: 0.0772 - learning_rate: 0.0010
Epoch 3/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 279ms/step - dice_coef: 0.9344 - iou_metric: 0.8900 - loss: 0.0656 - val_dice_coef: 0.9449 - val_iou_metric: 0.9129 - val_loss: 0.0551 - learning_rate: 0.0010
Epoch 4/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 286ms/step - dice_coef: 0.9388 - iou_metric: 0.8937 - loss: 0.0612 - val_dice_coef: 0.9533 - val_iou_metric: 0.9166 - val_loss: 0.0468 - learning_rate: 0.0010
Epoch 5/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 281ms/step - dice_coef: 0.9441 - iou_metric: 0.9005 - loss: 0.0559 - val_dice_coef: 0.9603 - val_iou_metric: 0.9265 - val_loss: 0.0400 - learning_rate: 0.0010
Epoch 